# Criterios de Partición en Árboles de Clasificación
**Dataset:** Iris (Fisher, 1936)

|                |   |
:----------------|---|
| **Nombre**     | Jorge Oviedo Magaña |
| **Fecha**      | 08/05/2026 |
| **Expediente** | 757048 |

---
## Motivación: ¿por qué no usar varianza en clasificación?

En los árboles de **regresión** el objetivo es minimizar la varianza conjunta de los nodos hijo:

$$\text{Impureza}(t) = \frac{1}{n}\sum_{i=1}^{n}(y_i - \bar{y})^2$$

Esto tiene sentido cuando la salida $y$ es un número real. En clasificación, la salida es una **categoría** (p. ej. *Setosa*, *Versicolor*, *Virginica*), por lo que la media no existe. Necesitamos medidas de **impureza** que capturen qué tan "mezcladas" están las clases en un nodo.

Las tres medidas más usadas son: **Gini**, **Entropía** y **Log Loss**.

---
## 1. Índice de Gini

### ¿Qué mide?
El índice de Gini mide la **probabilidad de clasificar incorrectamente** una observación elegida al azar, si la etiqueta se asignara según la distribución de clases del nodo.

### Fórmula

$$\text{Gini}(t) = 1 - \sum_{k=1}^{K} p_k^2$$

donde $p_k$ es la proporción de observaciones de la clase $k$ en el nodo $t$.

| Caso | Gini |
|------|------|
| Nodo puro (una sola clase) | 0 |
| Nodo perfectamente mezclado (binario) | 0.5 |

### Criterio de partición
Se evalúan todos los umbrales posibles y se elige el que **minimiza** el Gini ponderado de los dos hijos:

$$\text{Gini}_{\text{pond}}(t) = \frac{n_L}{n}\,\text{Gini}(L) + \frac{n_R}{n}\,\text{Gini}(R)$$

> **Ventaja:** Rápido de calcular (sin logaritmos). Es el criterio por defecto en `sklearn` (`criterion='gini'`).

---
## 2. Entropía (Ganancia de Información)

### ¿Qué mide?
La entropía proviene de la **teoría de la información** (Shannon, 1948). Mide el número promedio de bits necesarios para codificar la clase de una observación seleccionada al azar del nodo.

### Fórmula

$$H(t) = -\sum_{k=1}^{K} p_k \log_2(p_k)$$

| Caso | Entropía |
|------|----------|
| Nodo puro (una sola clase) | 0 bits |
| Nodo perfectamente mezclado (binario) | 1 bit |

### Criterio de partición: Ganancia de Información (IG)

$$IG = H(\text{padre}) - \left[\frac{n_L}{n}H(L) + \frac{n_R}{n}H(R)\right]$$

Se elige el umbral que **maximiza** $IG$, es decir, la mayor reducción de entropía. Equivale a minimizar la entropía ponderada de los hijos.

---
## 3. Log Loss (Pérdida Logarítmica)

### ¿Qué mide?
El Log Loss penaliza las predicciones **confiadas pero incorrectas**. A diferencia de la entropía de Shannon, el Log Loss evalúa la calidad de las **probabilidades predichas** para cada observación individual.

### Fórmula

$$\mathcal{L} = -\frac{1}{n}\sum_{i=1}^{n}\left[y_i \log(\hat{p}_i) + (1-y_i)\log(1-\hat{p}_i)\right]$$

donde $\hat{p}_i$ es la probabilidad estimada de que $y_i = 1$.

### Criterio de partición
Dentro de un nodo, $\hat{p}_i = \hat{p} = $ proporción de la clase positiva en ese nodo. El Log Loss ponderado de los hijos se minimiza.

---
## 4. ¿Cuál es la diferencia entre Entropía y Log Loss?

Aunque las fórmulas se parecen, existen diferencias conceptuales importantes:

| Aspecto | Entropía | Log Loss |
|---------|----------|----------|
| **Perspectiva** | Información promedio del nodo | Calidad de la probabilidad predicha por observación |
| **Escala** | Bits ($\log_2$) | Nats ($\log$ natural) |
| **Uso típico** | Criterio de partición en árboles (ID3, C4.5) | Función de pérdida en regresión logística, redes neuronales |
| **Penalización** | Distribución de clases en el nodo | Predicciones erróneas y confiadas ($\hat{p} \approx 0$ o $1$ incorrectas) |
| **Resultado** | Mismos umbrales óptimos que Log Loss (con $\log_2$ vs $\ln$, solo cambia la escala) | Escalado diferente, pero óptimos equivalentes en árboles |

> **Resumen:** Dentro de un árbol de decisión, Entropía y Log Loss seleccionan **el mismo umbral óptimo** (porque $\log_2 x = \ln x / \ln 2$ es una transformación lineal). La diferencia es conceptual: la entropía mide *incertidumbre del nodo*, el Log Loss mide *calidad de la probabilidad predicha por muestra*.

---
## Implementación: Dataset Iris

Usaremos el dataset **Iris** (Fisher, 1936), incluido en `sklearn`. Para crear un problema binario sencillo se conservan solo las clases *Versicolor* (0) y *Virginica* (1), y se usa el ancho del pétalo (`petal width`) como variable de partición.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.datasets import load_iris

# ── Cargar datos ────────────────────────────────────────────────────────────
iris = load_iris(as_frame=True)
df = iris.frame.copy()
df.columns = [*iris.feature_names, 'target']

# Problema binario: Versicolor (1→0) vs Virginica (2→1)
df = df[df['target'].isin([1, 2])].copy()
df['target'] = (df['target'] == 2).astype(int)

X = df['petal width (cm)'].values
y = df['target'].values

print(f'Total de muestras : {len(y)}')
print(f'Versicolor (0)    : {(y==0).sum()}')
print(f'Virginica  (1)    : {(y==1).sum()}')
print(f'Rango petal width : {X.min():.1f} – {X.max():.1f} cm')

---
## Funciones auxiliares

In [ ]:
def gini(y):
    """Gini = 1 - sum(pk^2)"""
    if len(y) == 0:
        return 0.0
    p = np.mean(y)
    return 1.0 - (p**2 + (1-p)**2)

def entropy(y):
    """Entropía de Shannon = -sum(pk * log2(pk))"""
    if len(y) == 0:
        return 0.0
    p = np.mean(y)
    if p in (0.0, 1.0):
        return 0.0
    return -(p * np.log2(p) + (1-p) * np.log2(1-p))

def log_loss_node(y):
    """Log Loss = -mean(yi*log(p_hat) + (1-yi)*log(1-p_hat)), p_hat = proporción del nodo"""
    if len(y) == 0:
        return 0.0
    p = np.clip(np.mean(y), 1e-15, 1 - 1e-15)
    return -(p * np.log(p) + (1-p) * np.log(1-p))

def weighted_impurity(metric_fn, y_left, y_right):
    """Métrica ponderada por tamaño de hijos"""
    n = len(y_left) + len(y_right)
    if n == 0:
        return 0.0
    return (len(y_left)/n) * metric_fn(y_left) + (len(y_right)/n) * metric_fn(y_right)

print('✓ Funciones definidas: gini, entropy, log_loss_node, weighted_impurity')

---
## Partición 1: Índice de Gini

Se evalúan todos los umbrales candidatos (valores únicos de `petal width`) y se elige el que **minimiza** el Gini ponderado de los dos nodos hijo.

In [ ]:
thresholds = np.unique(X)
gini_scores = [weighted_impurity(gini, y[X <= t], y[X > t]) for t in thresholds]

best_idx_g = np.argmin(gini_scores)
best_t_g   = thresholds[best_idx_g]
best_g     = gini_scores[best_idx_g]

L_g, R_g = y[X <= best_t_g], y[X > best_t_g]

print(f'Umbral óptimo (Gini) : petal_width <= {best_t_g:.2f} cm')
print(f'Gini ponderado mínimo: {best_g:.4f}')
print()
print(f'  Nodo izq (≤{best_t_g:.2f}): n={len(L_g):2d} | Versicolor={( L_g==0).sum():2d} | Virginica={( L_g==1).sum():2d} | Gini={gini(L_g):.4f}')
print(f'  Nodo der (> {best_t_g:.2f}): n={len(R_g):2d} | Versicolor={(R_g==0).sum():2d} | Virginica={(R_g==1).sum():2d} | Gini={gini(R_g):.4f}')

# Gráfica
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(thresholds, gini_scores, color='#1D9E75', linewidth=2)
axes[0].axvline(best_t_g, color='#E8593C', linestyle='--', label=f'Óptimo: {best_t_g:.2f} cm')
axes[0].scatter([best_t_g], [best_g], color='#E8593C', zorder=5, s=80)
axes[0].set_xlabel('Umbral petal width (cm)')
axes[0].set_ylabel('Gini ponderado')
axes[0].set_title('Índice de Gini por umbral')
axes[0].legend()

colors = ['#3B8BD4', '#E8593C']
labels = ['Versicolor (0)', 'Virginica (1)']
scatter_x = np.random.seed(42) or np.ones(len(L_g))*0.8 + np.random.uniform(-0.15, 0.15, len(L_g))
scatter_x2 = np.ones(len(R_g))*2.2 + np.random.uniform(-0.15, 0.15, len(R_g))
axes[1].scatter(np.ones(len(L_g))*0.8 + np.random.uniform(-0.15,0.15,len(L_g)), L_g, c=[colors[v] for v in L_g], alpha=0.7, s=60)
axes[1].scatter(np.ones(len(R_g))*2.2 + np.random.uniform(-0.15,0.15,len(R_g)), R_g, c=[colors[v] for v in R_g], alpha=0.7, s=60)
axes[1].set_xticks([0.8, 2.2]); axes[1].set_xticklabels([f'≤{best_t_g:.2f} cm', f'>{best_t_g:.2f} cm'])
axes[1].set_yticks([0,1]); axes[1].set_yticklabels(['Versicolor','Virginica'])
axes[1].set_title(f'Partición resultante (Gini)')
for lbl, c in zip(labels, colors):
    axes[1].scatter([], [], c=c, label=lbl)
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Partición 2: Entropía (Ganancia de Información)

Se calcula la entropía del nodo padre y la **ganancia de información** para cada umbral. Se selecciona el umbral que **maximiza** la IG.

In [ ]:
parent_H   = entropy(y)
ig_scores  = [parent_H - weighted_impurity(entropy, y[X <= t], y[X > t]) for t in thresholds]

best_idx_e = np.argmax(ig_scores)
best_t_e   = thresholds[best_idx_e]
best_ig    = ig_scores[best_idx_e]

L_e, R_e = y[X <= best_t_e], y[X > best_t_e]

print(f'Entropía del padre       : {parent_H:.4f} bits')
print(f'Umbral óptimo (Entropía) : petal_width <= {best_t_e:.2f} cm')
print(f'Ganancia de información  : {best_ig:.4f} bits')
print()
print(f'  Nodo izq (≤{best_t_e:.2f}): n={len(L_e):2d} | Versicolor={( L_e==0).sum():2d} | Virginica={( L_e==1).sum():2d} | H={entropy(L_e):.4f}')
print(f'  Nodo der (> {best_t_e:.2f}): n={len(R_e):2d} | Versicolor={(R_e==0).sum():2d} | Virginica={(R_e==1).sum():2d} | H={entropy(R_e):.4f}')

# Gráfica
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(thresholds, ig_scores, color='#3B8BD4', linewidth=2)
axes[0].axvline(best_t_e, color='#E8593C', linestyle='--', label=f'Óptimo: {best_t_e:.2f} cm')
axes[0].scatter([best_t_e], [best_ig], color='#E8593C', zorder=5, s=80)
axes[0].set_xlabel('Umbral petal width (cm)')
axes[0].set_ylabel('Ganancia de Información (bits)')
axes[0].set_title('Entropía — Ganancia de Información por umbral')
axes[0].legend()

axes[1].scatter(np.ones(len(L_e))*0.8 + np.random.uniform(-0.15,0.15,len(L_e)), L_e, c=[colors[v] for v in L_e], alpha=0.7, s=60)
axes[1].scatter(np.ones(len(R_e))*2.2 + np.random.uniform(-0.15,0.15,len(R_e)), R_e, c=[colors[v] for v in R_e], alpha=0.7, s=60)
axes[1].set_xticks([0.8, 2.2]); axes[1].set_xticklabels([f'≤{best_t_e:.2f} cm', f'>{best_t_e:.2f} cm'])
axes[1].set_yticks([0,1]); axes[1].set_yticklabels(['Versicolor','Virginica'])
axes[1].set_title('Partición resultante (Entropía)')
for lbl, c in zip(labels, colors):
    axes[1].scatter([], [], c=c, label=lbl)
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Partición 3: Log Loss

Se minimiza el Log Loss ponderado de los dos hijos. La diferencia respecto a la entropía es solo de escala (base del logaritmo: $\log_2$ vs $\ln$), por lo que suele encontrar el mismo umbral óptimo, pero ilustra la conexión con la función de pérdida usada en modelos probabilísticos.

In [ ]:
ll_scores  = [weighted_impurity(log_loss_node, y[X <= t], y[X > t]) for t in thresholds]

best_idx_l = np.argmin(ll_scores)
best_t_l   = thresholds[best_idx_l]
best_ll    = ll_scores[best_idx_l]

L_l, R_l = y[X <= best_t_l], y[X > best_t_l]

print(f'Umbral óptimo (Log Loss) : petal_width <= {best_t_l:.2f} cm')
print(f'Log Loss ponderado mín   : {best_ll:.4f}')
print()
print(f'  Nodo izq (≤{best_t_l:.2f}): n={len(L_l):2d} | Versicolor={( L_l==0).sum():2d} | Virginica={( L_l==1).sum():2d} | LL={log_loss_node(L_l):.4f}')
print(f'  Nodo der (> {best_t_l:.2f}): n={len(R_l):2d} | Versicolor={(R_l==0).sum():2d} | Virginica={(R_l==1).sum():2d} | LL={log_loss_node(R_l):.4f}')

# Gráfica
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(thresholds, ll_scores, color='#BA7517', linewidth=2)
axes[0].axvline(best_t_l, color='#E8593C', linestyle='--', label=f'Óptimo: {best_t_l:.2f} cm')
axes[0].scatter([best_t_l], [best_ll], color='#E8593C', zorder=5, s=80)
axes[0].set_xlabel('Umbral petal width (cm)')
axes[0].set_ylabel('Log Loss ponderado')
axes[0].set_title('Log Loss por umbral')
axes[0].legend()

axes[1].scatter(np.ones(len(L_l))*0.8 + np.random.uniform(-0.15,0.15,len(L_l)), L_l, c=[colors[v] for v in L_l], alpha=0.7, s=60)
axes[1].scatter(np.ones(len(R_l))*2.2 + np.random.uniform(-0.15,0.15,len(R_l)), R_l, c=[colors[v] for v in R_l], alpha=0.7, s=60)
axes[1].set_xticks([0.8, 2.2]); axes[1].set_xticklabels([f'≤{best_t_l:.2f} cm', f'>{best_t_l:.2f} cm'])
axes[1].set_yticks([0,1]); axes[1].set_yticklabels(['Versicolor','Virginica'])
axes[1].set_title('Partición resultante (Log Loss)')
for lbl, c in zip(labels, colors):
    axes[1].scatter([], [], c=c, label=lbl)
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Resumen comparativo

Se normalizan las tres curvas para visualizarlas en la misma escala y comparar cómo evolucionan con el umbral. Para Entropía se invierte la IG (se grafica la entropía ponderada) para que las tres vayan en la misma dirección (minimizar).

In [ ]:
def normalize(arr):
    arr = np.array(arr, dtype=float)
    mn, mx = arr.min(), arr.max()
    return (arr - mn) / (mx - mn) if mx > mn else arr - mn

ent_scores = [weighted_impurity(entropy, y[X<=t], y[X>t]) for t in thresholds]  # minimizar

plt.figure(figsize=(10, 4.5))
plt.plot(thresholds, normalize(gini_scores), color='#1D9E75', linewidth=2.5, label='Gini')
plt.plot(thresholds, normalize(ent_scores),  color='#3B8BD4', linewidth=2.5, linestyle='--', label='Entropía')
plt.plot(thresholds, normalize(ll_scores),   color='#BA7517', linewidth=2.5, linestyle=':',  label='Log Loss')

for t, c in [(best_t_g,'#1D9E75'), (best_t_e,'#3B8BD4'), (best_t_l,'#BA7517')]:
    plt.axvline(t, color=c, alpha=0.25, linewidth=8)

plt.xlabel('Umbral petal width (cm)', fontsize=12)
plt.ylabel('Métrica normalizada (menor = mejor)', fontsize=12)
plt.title('Comparación de criterios de impureza — Dataset Iris', fontsize=13)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

print('Umbral óptimo por criterio:')
print(f'  Gini     : petal_width <= {best_t_g:.2f} cm')
print(f'  Entropía : petal_width <= {best_t_e:.2f} cm')
print(f'  Log Loss : petal_width <= {best_t_l:.2f} cm')
print()
if best_t_g == best_t_e == best_t_l:
    print('→ Los tres criterios coinciden en el mismo umbral óptimo.')
else:
    print('→ Los criterios difieren en el umbral elegido (ver gráfica).')

---
## Conclusiones

1. **Gini** es la medida más rápida de calcular (sin logaritmos) y es el criterio por defecto en `sklearn`. Favorece particiones que aíslan la clase más frecuente.

2. **Entropía** proviene de la teoría de la información. Tiende a producir árboles ligeramente más balanceados que Gini y penaliza más las impurezas intermedias.

3. **Log Loss** como criterio de árbol equivale a la entropía escalada por $1/\ln 2$; en la práctica elige el **mismo umbral óptimo**. Su mayor utilidad es como función de pérdida en modelos probabilísticos (regresión logística, XGBoost) donde $\hat{p}$ proviene de un modelo, no de la proporción del nodo.

4. La diferencia clave entre **Entropía y Log Loss** es la perspectiva: la entropía mide la *incertidumbre del nodo completo*, mientras que el Log Loss mide la *calidad de la probabilidad predicha por cada observación individual*. Dentro de un árbol de decisión, ambas coinciden en la partición elegida.